In [1]:
import torch
from transformers import AutoModelForImageClassification

In [2]:
MODEL_ID = "facebook/convnext-tiny-224"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForImageClassification.from_pretrained(MODEL_ID
).to(device)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  114MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

ConvNextForImageClassification(
  (convnext): ConvNextModel(
    (embeddings): ConvNextEmbeddings(
      (patch_embeddings): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (layernorm): ConvNextLayerNorm((96,), eps=1e-06, elementwise_affine=True)
    )
    (encoder): ConvNextEncoder(
      (stages): ModuleList(
        (0): ConvNextStage(
          (downsampling_layer): ModuleList()
          (layers): ModuleList(
            (0-2): 3 x ConvNextLayer(
              (dwconv): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
              (layernorm): ConvNextLayerNorm((96,), eps=1e-06, elementwise_affine=True)
              (pwconv1): Linear(in_features=96, out_features=384, bias=True)
              (act): GELUActivation()
              (pwconv2): Linear(in_features=384, out_features=96, bias=True)
              (drop_path): Identity()
            )
          )
        )
        (1): ConvNextStage(
          (downsampling_layer): ModuleList(
          

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

In [3]:
print("Device:", device)
print("Model:", MODEL_ID)
print("Model class:", model.__class__.__name__)

Device: cuda
Model: facebook/convnext-tiny-224
Model class: ConvNextForImageClassification


**model configuration**

In [4]:
config = model.config

print("Image size              :", config.image_size)
print("Patch size              :", config.patch_size)
print("Number of stages        :", config.num_stages)
print("Stage channels          :", config.hidden_sizes)
print("Blocks per stage        :", config.depths)
print("Activation              :", config.hidden_act)
print("Layer scale init value  :", config.layer_scale_init_value)
print("Number of labels        :", config.num_labels)

Image size              : 224
Patch size              : 4
Number of stages        : 4
Stage channels          : [96, 192, 384, 768]
Blocks per stage        : [3, 3, 9, 3]
Activation              : gelu
Layer scale init value  : 1e-06
Number of labels        : 1000


**Parameter X-ray**

In [5]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

classifier_parameters = sum(
    parameter.numel()
    for parameter in model.classifier.parameters()
)

print(f"Total parameters     : {total_parameters:,}")
print(f"Trainable parameters : {trainable_parameters:,}")
print(f"Classifier parameters: {classifier_parameters:,}")

Total parameters     : 28,589,128
Trainable parameters : 28,589,128
Classifier parameters: 769,000


**Exact tensor-shape**

In [6]:
pixel_values = torch.randn(1, 3, 224, 224, device=device)

with torch.inference_mode():
    outputs = model(
        pixel_values=pixel_values,
        output_hidden_states=True,
        return_dict=True
    )

print("Number of hidden states:", len(outputs.hidden_states))
for index, hidden_state in enumerate(outputs.hidden_states):
    print(
        f"Hidden state {index}:",
        tuple(hidden_state.shape)
    )

print("Logits:", tuple(outputs.logits.shape))

Number of hidden states: 5
Hidden state 0: (1, 96, 56, 56)
Hidden state 1: (1, 96, 56, 56)
Hidden state 2: (1, 192, 28, 28)
Hidden state 3: (1, 384, 14, 14)
Hidden state 4: (1, 768, 7, 7)
Logits: (1, 1000)
